# J4-B — Adapter un modèle au résumé puis vérifier les faits

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 4 · 80 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Fine-tuner un petit modèle causal pour résumer des comptes rendus.
- Comparer extractif, modèle initial et adaptateur sur les mêmes sources tenues à part.
- Séparer recouvrement lexical, fidélité factuelle et utilité.

## Parcours de l’après-midi

| Étape | Minutes |
|---|---:|
| Données, format et baseline | 20 |
| LoRA borné au résumé | 25 |
| Comparaison et audit des faits | 20 |
| Erreurs, export et restitution | 15 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'datasets': '4.1.1', 'peft': '0.17.1', 'trl': '0.23.1', 'rouge-score': '0.1.2'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
versions["torch"] = torch.__version__
results["device"] = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print("Périphérique :", results["device"])
if DEVICE == "cuda":
    print("VRAM totale (Gio) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("Pas de GPU : les sections indiquées restent utilisables ; ne pas présenter un modèle non entraîné comme adapté.")


## 1. Une tâche de fine-tuning différente de la réponse au client

Nous adaptons un modèle **au résumé**. Cet adaptateur est distinct du SFT support du TP05.
Le corpus original comporte 16 comptes rendus train, 2 validation et 4 test. Aucun texte ni scénario
n’est partagé entre les partitions. Ce volume sert à comprendre l’adaptation ; il ne suffit pas
pour prouver une qualité métier et peut faire surapprendre le style des références.

Le résumé doit préserver le problème, l’action effectuée et l’incertitude. Transformer
« remboursement non confirmé » en « remboursement effectué » est une erreur importante.
Une baseline extrait les deux premières phrases ; le modèle initial puis adapté génèrent un résumé.

Sans GPU, les données, l’extractif et ROUGE restent exécutables. Le fine-tuning est alors déclaré
non exécuté. Les 12 pas choisis bornent l’exercice ; leur durée dépend du runtime et n’est pas certifiée.


In [ ]:
cases = [{'id': 's01', 'split': 'validation', 'text': "La commande OR-301 comporte deux livres. Le suivi est resté au centre de tri de Nantes depuis mardi. Le client a déjà vérifié son adresse et n'a reçu aucun avis de passage. Le support a ouvert une demande auprès du transporteur. Aucune date d'arrivée n'est confirmée.", 'reference': "La commande OR-301 de deux livres est bloquée au tri de Nantes depuis mardi. Le support a sollicité le transporteur ; aucune date de livraison n'est confirmée."}, {'id': 's02', 'split': 'validation', 'text': "Une cliente voit deux opérations de 42 euros pour la commande OR-302. L'une est un débit et l'autre une autorisation bancaire en attente. Le support lui a demandé de vérifier l'évolution auprès de sa banque. Aucun second débit définitif n'a été établi. Le mot de passe n'a pas été demandé.", 'reference': "Pour OR-302, les deux opérations de 42 euros correspondent à un débit et à une autorisation en attente. La cliente doit vérifier auprès de sa banque ; aucun second débit n'est confirmé."}, {'id': 's03', 'split': 'test', 'text': "Le retour RT-401 contient une veste et a été reçu jeudi. Son contrôle n'est pas encore terminé. Le client demande quand le remboursement sera fait. Le support ne dispose pas de délai confirmé et a transmis la question à l'équipe retour. Aucune somme n'a été remboursée à ce stade.", 'reference': "La veste du retour RT-401 a été reçue jeudi et reste en contrôle. Le support a interrogé l'équipe retour ; aucun remboursement ni délai n'est confirmé."}, {'id': 's04', 'split': 'test', 'text': "L'application version 2.4 se ferme à l'ouverture du panier sur un téléphone Android. Le problème persiste après redémarrage. La cliente a transmis les étapes et une capture sans données personnelles. Le ticket BUG-402 est envoyé à l'équipe technique. Le site web reste accessible.", 'reference': "Le ticket BUG-402 concerne un plantage du panier dans l'application 2.4 sur Android malgré un redémarrage. L'équipe technique dispose des étapes et d'une capture ; le site web reste accessible."}, {'id': 's05', 'split': 'test', 'text': "Le colis OR-403 est indiqué livré à un voisin. Le client a vérifié auprès de ce voisin qui confirme ne rien avoir reçu. Le support a demandé une preuve de remise au transporteur. L'enquête est en cours. Il n'y a aucune confirmation de perte ni de nouvelle expédition.", 'reference': 'Le colis OR-403 est marqué livré à un voisin qui nie sa réception. Le support attend une preuve de remise ; ni perte ni réexpédition ne sont confirmées.'}, {'id': 's06', 'split': 'test', 'text': "Une utilisatrice demande la suppression de son compte CP-404. Elle a rempli le formulaire officiel mais la vérification d'identité n'est pas terminée. Le support a confirmé la réception de sa demande. Les données ne sont pas encore supprimées. La prochaine étape est de terminer la vérification dans le canal officiel.", 'reference': "La demande de suppression CP-404 est reçue, mais la vérification d'identité reste à terminer par le canal officiel. Les données ne sont pas encore supprimées."}]
SUMMARY_TRAIN = [('TR-01', "La commande VE-101 contient une lampe. Le colis est arrivé mais le câble manque. La cliente a envoyé une photo du contenu. Le support a demandé une vérification à l'entrepôt. Aucun nouvel envoi n'est confirmé.", "La lampe VE-101 est arrivée sans câble. Le support vérifie le contenu auprès de l'entrepôt après réception d'une photo ; aucun renvoi n'est confirmé."), ('TR-02', 'Le client a reçu une facture avec son ancien nom de société. Le document porte FC-102. La comptabilité examine la demande de correction. Le client ne doit pas modifier lui-même le fichier.', "La facture FC-102 comporte l'ancien nom de société. La comptabilité examine une correction ; le client ne doit pas modifier le document."), ('TR-03', 'Le ticket CO-103 concerne un lien de connexion expiré. Le client a demandé un nouveau lien et a réussi à ouvrir son compte. Le support a clos le ticket après sa confirmation.', "Le lien de connexion CO-103 avait expiré. Un nouveau lien a permis l'accès au compte et le ticket est clos après confirmation."), ('TR-04', "Le retour RE-104 est arrivé sans numéro de commande. L'équipe demande au client la référence d'achat pour identifier le produit. Le remboursement n'a pas été déclenché.", "Le retour RE-104 doit être identifié avec la référence d'achat demandée au client. Aucun remboursement n'a été déclenché."), ('TR-05', "Sur Firefox, le menu mobile reste fermé. Le ticket TE-105 contient une capture et les étapes. L'équipe technique a reproduit le problème. Aucun correctif n'est encore déployé.", "Le menu mobile ne s'ouvre pas dans Firefox pour TE-105. L'équipe a reproduit le problème ; le correctif reste à déployer."), ('TR-06', 'La commande VE-106 a été divisée en deux envois. Un seul colis a été reçu. Le support a transmis le suivi du second. Le client confirme pouvoir consulter les deux références.', 'VE-106 comprend deux colis dont un reçu. Le support a fourni le second suivi, désormais accessible au client.'), ('TR-07', "Le paiement PA-107 a été refusé trois fois. Aucune commande n'a été validée. Le client contacte sa banque. Le support ne connaît pas la cause bancaire du refus.", 'Trois paiements PA-107 ont été refusés sans commande validée. Le client consulte sa banque ; le support ne connaît pas la cause.'), ('TR-08', "Une cliente a perdu son appareil d'authentification. Le dossier CO-108 est transmis au service de récupération de compte. La vérification d'identité doit encore être effectuée.", "CO-108 concerne la perte de l'appareil d'authentification. La récupération est transmise au service concerné, avec vérification d'identité encore nécessaire."), ('TR-09', "Pour RE-109, le client souhaite un échange contre une taille supérieure. La taille demandée n'est pas en stock aujourd'hui. Le support a expliqué les autres options sans confirmer de date de réassort.", "L'échange RE-109 est limité par l'absence de la taille souhaitée. Le support a présenté les autres options ; aucune date de réassort n'est confirmée."), ('TR-10', "La page catalogue produit une erreur 500. Le ticket TE-110 a été ouvert à 14 heures. L'équipe technique a redémarré le service et vérifié le chargement. Le client confirme que la page fonctionne.", "L'erreur 500 de TE-110 a été résolue après redémarrage du service. Le chargement a été vérifié et le client confirme le fonctionnement."), ('TR-11', "Le colis VE-111 est endommagé à l'extérieur. Le produit n'a pas encore été déballé. Le support demande des photos avant de décider de la suite. La présence d'un dommage sur le produit est inconnue.", "L'emballage VE-111 est abîmé, mais l'état du produit reste inconnu. Le support attend des photos avant toute décision."), ('TR-12', "Le ticket FC-112 concerne une facture absente de l'espace client. Le paiement a bien été validé. Le support a transmis le document par le canal officiel. Le client en confirme la réception.", "La facture FC-112 manquait malgré un paiement validé. Le support a envoyé le document par le canal officiel et le client l'a reçu."), ('TR-13', "Une connexion inconnue a été signalée dans CO-113. Le client a changé son mot de passe et fermé les sessions actives. Le support a transmis le signalement pour examen. Une intrusion n'est pas confirmée.", "Pour CO-113, le client a changé son mot de passe et fermé ses sessions après une connexion inconnue. Le signalement est examiné ; aucune intrusion n'est confirmée."), ('TR-14', "Le retour RE-114 a été refusé au relais car le paquet dépasse les dimensions. Le support recherche un autre mode de collecte. Le client conserve le colis et n'a pas payé un nouvel envoi.", 'Le relais a refusé RE-114 en raison des dimensions. Le support cherche une collecte adaptée ; le client garde le colis sans avancer de frais.'), ('TR-15', "Le formulaire TE-115 affiche une erreur avec les noms accentués. L'équipe a confirmé un défaut de validation. Un correctif est en préparation. Aucune date de mise en ligne n'est donnée.", "TE-115 révèle un défaut de validation des noms accentués confirmé par l'équipe. Un correctif est préparé, sans date de mise en ligne."), ('TR-16', "Le client a demandé une nouvelle adresse pour VE-116 après l'expédition. Le transporteur ne propose pas de redirection sur ce suivi. Le support étudie la situation. La modification n'est pas confirmée.", "Pour VE-116 déjà expédiée, aucune redirection n'est proposée par le transporteur. Le support examine la demande ; le changement d'adresse n'est pas confirmé.")]
import re
train_cases = [{"id": ident, "split": "train", "text": text, "reference": ref}
               for ident, text, ref in SUMMARY_TRAIN]
all_cases = train_cases + cases
assert len(set(c["id"] for c in all_cases)) == len(all_cases)
assert len(set(c["text"] for c in all_cases)) == len(all_cases)
assert len(set(c["reference"] for c in all_cases)) == len(all_cases)
def extractive_summary(text):
    return " ".join(re.split(r"(?<=[.!?])\s+", text)[:2])
for case in cases: case["extractif"] = extractive_summary(case["text"])
display(pd.DataFrame(train_cases[:3]))
print(pd.Series([c["split"] for c in all_cases]).value_counts())


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from contextlib import nullcontext
RUN_TRAINING = DEVICE == "cuda"
MAX_STEPS = 12
RANK = 8
MAX_LENGTH = 512
SYSTEM = "Tu résumes uniquement les faits présents dans le texte fourni."
PROMPT = "Résume ce compte rendu en deux phrases en français. Préserve la référence, le problème, l'action et ce qui n'est pas confirmé. N'ajoute aucune information. Texte : "
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
def as_summary_pair(case):
    return {"prompt": [{"role": "system", "content": SYSTEM},
                       {"role": "user", "content": PROMPT + case["text"]}],
            "completion": [{"role": "assistant", "content": case["reference"]}]}
lengths = [len(tokenizer.apply_chat_template(as_summary_pair(c)["prompt"] + as_summary_pair(c)["completion"],
                                            tokenize=True)) for c in all_cases]
print("Longueur max sérialisée :", max(lengths))
assert max(lengths) <= MAX_LENGTH, "Ne pas tronquer silencieusement une réponse cible."
model = None
if RUN_TRAINING:
    set_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE).eval()
def summarize(text, initial=False):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": PROMPT + text}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(DEVICE)
    context = model.disable_adapter() if initial and hasattr(model, "peft_config") else nullcontext()
    model.eval()
    with context, torch.inference_mode():
        output = model.generate(**inputs, do_sample=False, max_new_tokens=110,
                                pad_token_id=tokenizer.pad_token_id, use_cache=True)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
if model is not None:
    for case in cases:
        if case["split"] == "validation": case["avant"] = summarize(case["text"], initial=True)
display(pd.DataFrame([c for c in cases if c["split"] == "validation"]))


## 2. Adapter seulement de petites matrices LoRA

Lire les deux sorties de validation avant de lancer. Figer le prompt et le budget de 12 pas.
Les poids du modèle initial resteront gelés. Pour un nouveau réglage, repartir du modèle initial,
jamais du modèle déjà adapté, et sélectionner uniquement sur validation.

Le format prompt/completion supervise seulement le résumé cible. Vérifier les tokens ignorés
et appris dans le vrai batch du collator. Le T4 et le L4 utilisent ici fp16 et bf16=False.
Cette expérience n’importe pas l’adaptateur support : elle apprend une autre tâche.


In [ ]:
summary_trainer = None
if model is not None:
    lora = LoraConfig(r=RANK, lora_alpha=16, lora_dropout=0.05,
                      target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora)
    model.config.use_cache = False
    model.print_trainable_parameters()
    summary_ds = {split: Dataset.from_list([as_summary_pair(c) for c in all_cases if c["split"] == split])
                  for split in ["train", "validation"]}
    summary_config = SFTConfig(output_dir="checkpoints_resume", max_steps=MAX_STEPS,
        per_device_train_batch_size=1, gradient_accumulation_steps=4, per_device_eval_batch_size=1,
        learning_rate=1e-4, max_length=MAX_LENGTH, packing=False,
        completion_only_loss=True, assistant_only_loss=False, eos_token="<|im_end|>",
        fp16=True, bf16=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps", eval_steps=6, save_strategy="no", logging_steps=3,
        report_to="none", seed=SEED, data_seed=SEED, dataloader_num_workers=0,
        push_to_hub=False, optim="adamw_torch")
    summary_trainer = SFTTrainer(model=model, args=summary_config,
        train_dataset=summary_ds["train"], eval_dataset=summary_ds["validation"], processing_class=tokenizer)
    batch = summary_trainer.data_collator([summary_trainer.train_dataset[0]])
    labels = batch["labels"][0]
    assert (labels == -100).any() and (labels != -100).any()
    print("CIBLE SUPERVISÉE :", tokenizer.decode(labels[labels != -100]))
    started = time.perf_counter()
    summary_trainer.train()
    results["summary_training"] = {"seconds": time.perf_counter() - started, "steps": MAX_STEPS,
        "rank": RANK, "model": MODEL_ID, "train": 16, "validation": 2, "test": 4,
        "validation_metrics": summary_trainer.evaluate(), "history": summary_trainer.state.log_history}
    for case in cases:
        if case["split"] == "validation":
            case["avant"] = summarize(case["text"], initial=True)
            case["apres"] = summarize(case["text"])
    display(pd.DataFrame([c for c in cases if c["split"] == "validation"]))
else:
    results["summary_training"] = None
    print("Fine-tuning résumé non exécuté : aucune sortie adaptée à déclarer.")


## 3. Test tenu à part : extractif / initial / adapté

Figer les réglages puis évaluer les quatre mêmes comptes rendus avec les trois méthodes.
Le modèle initial est obtenu en désactivant l’adaptateur, sans modifier les poids de base.
Conserver le même prompt, la même limite de 110 tokens et le décodage glouton.
Une comparaison utile doit accepter un résultat négatif.

Grille humaine : fidélité 0–2, couverture des faits utiles 0–2, concision 0–2.
Chaque affirmation doit pointer vers une phrase source. Une invention importante vaut 0 en fidélité,
même si la sortie est fluide. Faire noter par l’autre binôme en masquant le nom de méthode.


In [ ]:
choix_prompt = "À compléter : configuration figée, observation de validation, hypothèse avant test."
if model is not None:
    for case in cases:
        if case["split"] == "test":
            case["avant"] = summarize(case["text"], initial=True)
            case["apres"] = summarize(case["text"])
test_cases = [c for c in cases if c["split"] == "test"]
display(pd.DataFrame(test_cases))


## 4. ROUGE-L ne juge pas la vérité

ROUGE-L mesure le recouvrement par sous-séquence commune. Nous utilisons un tokenizer Unicode
simple qui conserve les accents, sans stemming. Ce protocole n’est pas directement comparable
à tous les scores ROUGE publiés. Une référence unique peut pénaliser une paraphrase valide.

Mesurer aussi les deux phrases qui ne diffèrent que par la négation. Expliquer pourquoi leur score
lexical ne suffit pas pour décider laquelle respecte la source.


In [ ]:
from rouge_score import rouge_scorer
class FrenchTokenizer:
    def tokenize(self, text):
        return re.findall(r"\w+", text.casefold(), flags=re.UNICODE)
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False, tokenizer=FrenchTokenizer())
metric_rows = []
methods = ["extractif", "avant", "apres"]
for case in test_cases:
    for method in methods:
        if method in case:
            score = scorer.score(case["reference"], case[method])["rougeL"]
            metric_rows.append({"id": case["id"], "methode": method, "rougeL_f1": score.fmeasure,
                                "mots": len(FrenchTokenizer().tokenize(case[method]))})
display(pd.DataFrame(metric_rows))
display(pd.DataFrame(metric_rows).groupby("methode")[["rougeL_f1", "mots"]].mean())
reference = "Le remboursement de la commande n'est pas confirmé."
candidates = [reference, "Le remboursement de la commande est confirmé."]
display(pd.DataFrame({"candidat": candidates,
                      "rougeL": [scorer.score(reference, text)["rougeL"].fmeasure for text in candidates]}))
human_grid = pd.DataFrame([{"id": c["id"], "methode": method, "fidelite_0_2": None,
    "couverture_0_2": None, "concision_0_2": None, "preuve_source": "À compléter"}
    for c in test_cases for method in methods if method in c])
display(human_grid)
human_grid.to_csv(OUTPUT / "j4_grille_resume.csv", index=False)
results.update({"model": MODEL_ID, "prompt": PROMPT, "choix_prompt": choix_prompt,
    "cases": cases, "metrics": metric_rows, "human_grid": human_grid.to_dict(orient="records")})


## 5. Exporter l’adaptateur de résumé

Télécharger adaptateur_resume.zip avant de fermer le runtime. Il contient les petites matrices
adaptées, le tokenizer et la provenance ; les poids du modèle Qwen de base doivent être chargés
séparément. Ce fichier est différent de adaptateur_lora.zip, entraîné au support dans le TP05.

Restituer un exemple fidèle, une omission ou invention et une décision : garder la baseline,
le modèle initial ou l’adaptateur. Indiquer quelles données nouvelles permettraient une autre conclusion.


In [ ]:
import shutil
if model is not None:
    adapter_dir = Path("adaptateur_resume")
    model.save_pretrained(adapter_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_dir)
    (adapter_dir / "provenance.json").write_text(json.dumps({"base_model": MODEL_ID,
        "task": "résumé de comptes rendus fictifs français", "versions": versions,
        "training": results["summary_training"], "train_ids": [c["id"] for c in train_cases],
        "validation_ids": [c["id"] for c in cases if c["split"] == "validation"],
        "test_ids": [c["id"] for c in test_cases]}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (adapter_dir / "README.md").write_text(
        "---\nbase_model: " + MODEL_ID + "\nlibrary_name: peft\nlanguage: fr\nlicense: apache-2.0\n---\n"
        "# Adaptateur pédagogique de résumé\n\n16 comptes rendus fictifs train, 2 validation, 4 test. "
        "Aucune garantie de fidélité ni de performance métier. Consulter provenance.json.\n", encoding="utf-8")
    assert (adapter_dir / "adapter_config.json").exists()
    assert (adapter_dir / "adapter_model.safetensors").exists()
    print("À télécharger :", shutil.make_archive("adaptateur_resume", "zip", root_dir=adapter_dir))
else:
    print("Pas d'adaptateur de résumé : entraînement non exécuté.")
conclusion_binome = "À compléter : comparaison réellement observée, fidélité, limite du jeu de test et prochaine expérience."


## Ressources pour prolonger

- [HF — Résumé](https://huggingface.co/learn/llm-course/fr/chapter7/5)
- [TRL 0.23.1 — SFT et loss de completion](https://huggingface.co/docs/trl/v0.23.1/en/sft_trainer)
- [Google Research — ROUGE](https://github.com/google-research/google-research/tree/master/rouge)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '06_j4_resume_evaluation'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j4_resume_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
